In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2Model

torch.set_grad_enabled(False)

model = GPT2Model.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [84]:
#Embeddings

texts = ["Hello, good morning"]


def get_embeddings(input):
    wpe = model.wpe.weight
    wte = model.wte.weight
    encoded = tokenizer(input, return_tensors="pt")
    ids_input = encoded["input_ids"]
    num_tokens = ids_input.shape[1]
    positions = torch.arange(num_tokens)
    embeddings = wpe[positions] + wte[ids_input]

    return embeddings

embeddings = get_embeddings(texts)


In [4]:
#Huggingface embeddings results

salida_modelo = model(ids_input, output_hidden_states=True)

embeddings_modelo = salida_modelo.hidden_states[0]

print(embeddings_modelo)

tensor([[[-0.0875, -0.3301,  0.0152,  ...,  0.0285, -0.0014,  0.0068],
         [ 0.0355, -0.0567, -0.0626,  ...,  0.0619, -0.0195, -0.0601],
         [-0.0796, -0.1353,  0.0913,  ..., -0.1339, -0.0702, -0.1327],
         [ 0.0633,  0.0460,  0.1728,  ...,  0.0879,  0.0270, -0.1788]]])


In [99]:
def comparar(tensor_1, tensor_2):
    diferencias = (tensor_1 - tensor_2).abs()
    diferencia_max = diferencias.max().item()

    print(f"Diferencia máxima: {diferencia_max}")

    torch.testing.assert_close(tensor_1, tensor_2,atol=1e-4, rtol= 1e-5 )



In [6]:
#LayerNorm

def layernorm(tensor,weight,bias):
    mean = torch.mean(tensor, dim=-1, keepdim=True)
    eps = 1e-5
    variance = torch.var(tensor, dim=-1, keepdim=True, correction=0)

    output = ((tensor-mean)/torch.sqrt(variance+eps))* weight + bias
    return output


In [7]:
mi_resultado_layernorm = layernorm(embeddings,model.h[0].ln_1.weight,model.h[0].ln_1.bias)

In [8]:
#Huggingface h[0] ln_1 results

resultado_oficial = model.h[0].ln_1(embeddings_modelo)

comparar(mi_resultado_layernorm, resultado_oficial)

Diferencia máxima: 5.960464477539063e-08


In [18]:
#MLP Implementation
import torch.nn.functional as fu

def mlp(tensor, weight_1, bias_1, weight_2, bias_2):
    inter = tensor @ weight_1 + bias_1
    inter_2 = fu.gelu(inter, approximate="tanh")
    output = inter_2 @ weight_2 + bias_2

    return output



In [19]:
mi_resultado_mlp = mlp(embeddings, model.h[0].mlp.c_fc.weight, model.h[0].mlp.c_fc.bias, model.h[0].mlp.c_proj.weight, model.h[0].mlp.c_proj.bias)
resultado_oficial_2 = model.h[0].mlp(embeddings)

comparar(mi_resultado_mlp, resultado_oficial_2)


Diferencia máxima: 1.52587890625e-05


In [46]:
t = torch.arange(12.).reshape(2, 6)
a, b, c = t.split(2, dim=-1)
u = t.view(2, 3, 2)
print(u.transpose(0, 1).shape)

mask = torch.tril(torch.ones(4, 4))
s = torch.zeros(4, 4).masked_fill(mask == 0, float("-inf"))
print(torch.softmax(s, dim=-1))


mask_2 = torch.tril(torch.ones(6,6))
s_2 = torch.zeros(6,6).masked_fill(mask_2==0, float("-inf"))
print(s_2)
print(torch.softmax(s_2, dim=-1))

torch.Size([3, 2, 2])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667]])


In [50]:
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [75]:
#Implementing the attention layer

def attention(tensor, weight_1, bias_1, weight_2, bias_2):
    n_heads = 12
    head_dim = 768 // n_heads
    q,k,v = ((tensor @ weight_1) + bias_1).split(768, dim=-1)
    q = q.view(*q.shape[:-1],n_heads,head_dim)
    q = q.transpose(-3,-2)
    k = k.view(*k.shape[:-1],n_heads,head_dim)
    k = k.transpose(-3,-2)
    v = v.view(*v.shape[:-1],n_heads,head_dim)
    v = v.transpose(-3,-2)

    numerator = q @ k.transpose(-2,-1)
    denominator = head_dim ** 0.5
    mask = torch.tril(torch.ones(k[0].shape[-2],k[0].shape[-2], device=tensor.device))
    inter_mask = (numerator/denominator).masked_fill(mask==0, float("-inf"))
    softmax_res = torch.softmax(inter_mask, dim=-1)
    output_1 = softmax_res @ v

    output_1 = output_1.transpose(-3,-2)
    output_1 = output_1.reshape(*output_1.shape[:-2],n_heads*head_dim)

    output_2 = output_1@weight_2 + bias_2

    return output_2

In [76]:
mi_resultado_attention = attention(embeddings, model.h[0].attn.c_attn.weight, model.h[0].attn.c_attn.bias, model.h[0].attn.c_proj.weight, model.h[0].attn.c_proj.bias)

attention_oficial = model.h[0].attn(embeddings)[0]


comparar(mi_resultado_attention,attention_oficial)

Diferencia máxima: 1.71661376953125e-05


In [82]:
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [77]:
#Full Block Implementation

def full_layer_block(tensor, w1, b1, w2, b2, w3, b3, w4, b4, w5, b5, w6, b6):

    inter_1 = layernorm(tensor,w1,b1)
    inter_1 = attention(inter_1,w2,b2,w3,b3)

    res_1 = tensor + inter_1

    inter_2 = layernorm(res_1,w4,b4)
    inter_2 = mlp(inter_2,w5,b5,w6,b6)

    res_2 = res_1 + inter_2

    return res_2


In [81]:
mi_resultado_bloque = full_layer_block(embeddings,model.h[0].ln_1.weight,model.h[0].ln_1.bias, model.h[0].attn.c_attn.weight, model.h[0].attn.c_attn.bias, model.h[0].attn.c_proj.weight, model.h[0].attn.c_proj.bias,model.h[0].ln_2.weight, model.h[0].ln_2.bias, model.h[0].mlp.c_fc.weight, model.h[0].mlp.c_fc.bias, model.h[0].mlp.c_proj.weight, model.h[0].mlp.c_proj.bias )

result_off_bloque = model.h[0](embeddings)

comparar(mi_resultado_bloque, result_off_bloque)

Diferencia máxima: 7.62939453125e-06


In [85]:
def gpt2_model(input_text):

    embeddings = get_embeddings(input_text)
    for layer in range (12):
        embeddings = full_layer_block(embeddings,model.h[layer].ln_1.weight,model.h[layer].ln_1.bias, model.h[layer].attn.c_attn.weight, model.h[layer].attn.c_attn.bias, model.h[layer].attn.c_proj.weight, model.h[layer].attn.c_proj.bias,model.h[layer].ln_2.weight, model.h[layer].ln_2.bias, model.h[layer].mlp.c_fc.weight, model.h[layer].mlp.c_fc.bias, model.h[layer].mlp.c_proj.weight, model.h[layer].mlp.c_proj.bias)

    embeddings = layernorm(embeddings, model.ln_f.weight, model.ln_f.bias)

    return embeddings


In [100]:
input = ["Hello"]

wpe = model.wpe.weight
wte = model.wte.weight
encoded = tokenizer(input, return_tensors="pt")
ids_input = encoded["input_ids"]


mi_resultado_gpt = gpt2_model("Hello")

gpt_oficial = model(ids_input).last_hidden_state

comparar(mi_resultado_gpt,gpt_oficial)

Diferencia máxima: 9.1552734375e-05
